# Google Rank Tracking - Ad-Hoc Exploration

This notebook is for exploratory analysis of ranking data.

Use this alongside the MCP approach for deep dives.

In [ ]:
# Setup
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.append(str(ROOT / 'production_code'))
sys.path.append(str(ROOT / 'analysis'))

from database import RankTrackerDB
from trend_analysis import TrendAnalyzer
import json
from datetime import date, timedelta

In [ ]:
# Connect to database
db = RankTrackerDB('../production_code/rank_tracker.db')
analyzer = TrendAnalyzer(db)

# See what we're tracking
keywords = db.get_keywords()
print(f"Tracking {len(keywords)} keywords:")
for kw in keywords:
    print(f"  - {kw.keyword} → {kw.target_url} ({kw.location})")

## Quick Trend Check

Check how a specific keyword has performed over the last 30 days.

In [ ]:
# Analyze a specific keyword
keyword_to_check = "proxy api"  # Change this

matches = [k for k in keywords if k.keyword == keyword_to_check]
if not matches:
    print(f"Keyword '{keyword_to_check}' not found")
else:
    for kw in matches:
        trend = analyzer.get_keyword_trend(kw.id, days=30)
        if "error" in trend:
            print(f"{kw.target_url}: no data")
            continue
        print(f"Keyword: {keyword_to_check} → {kw.target_url}")
        print(f"  Current position: {trend['current']}")
        print(f"  Best position: {trend['best']}")
        print(f"  Worst position: {trend['worst']}")
        print(f"  Average: {trend['average']:.1f}")
        print(f"  Trend: {trend['trend']}")
        print(f"  Volatility: {trend['volatility']:.2f}")
        print()

## Position Distribution

See how often you rank in different position ranges.

In [ ]:
# Get all rankings for the last 30 days
from collections import Counter

all_positions = []
for kw in keywords:
    history = db.get_ranking_history(kw.id, days=30)
    all_positions.extend([r.position for r in history if r.position])

if all_positions:
    # Bucket positions
    buckets = {
        '1-3': len([p for p in all_positions if p <= 3]),
        '4-10': len([p for p in all_positions if 4 <= p <= 10]),
        '11-20': len([p for p in all_positions if 11 <= p <= 20]),
        '21-50': len([p for p in all_positions if 21 <= p <= 50]),
        '51-100': len([p for p in all_positions if 51 <= p <= 100]),
    }
    
    print("Position distribution (last 30 days):")
    total = len(all_positions)
    for bucket, count in buckets.items():
        pct = count / total * 100
        bar = '█' * int(pct / 2)
        print(f"  {bucket:>8}: {count:>3} ({pct:>5.1f}%) {bar}")
else:
    print("No ranking data available")

## Significant Changes

Find keywords with big ranking movements.

In [ ]:
# Check for significant changes
changes = db.get_position_changes(days=7)

if changes:
    print("Significant changes (last 7 days):")
    for c in changes:
        direction = "↑" if c['change'] > 0 else "↓"
        print(f"  {direction} {c['keyword']}: {c['prev_pos']} → {c['current_pos']} "
              f"({c['change']:+d})")
else:
    print("No significant changes in the last 7 days")

## Export for External Analysis

Export data for analysis in other tools.

In [ ]:
# Export all data to JSON
export_data = {
    'exported_at': date.today().isoformat(),
    'keywords': []
}

for kw in keywords:
    history = db.get_ranking_history(kw.id, days=90)
    export_data['keywords'].append({
        'keyword': kw.keyword,
        'target_url': kw.target_url,
        'location': kw.location,
        'history': [
            {'date': r.date.isoformat(), 'position': r.position}
            for r in history
        ]
    })

with open('ranking_export.json', 'w') as f:
    json.dump(export_data, f, indent=2)

print(f"Exported {len(export_data['keywords'])} keywords to ranking_export.json")